# 05 · Features, the job-request baseline B0, and placebos

1. **Window series.** The readings of each channel in `[t0, t0 + 20 min)` are averaged over the
   job's nodes, per cadence interval counted from the job's start, without interpolation.
2. **Leakage check V1.** No reading may be later than `t0 + 20 min`.
3. **Features.** Each of the ten groups becomes four columns: minimum, maximum, mean and standard
   deviation of its series in the window.
4. **B0.** Only what the user requested at submission.
5. **Placebos.** A real group's four columns with the rows permuted.

Output: `data/features.parquet`, one row per GPU job.

In [1]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from tcc.config import (B0_NUMERIC, DATA, GROUPS, IPMI_GROUPS, N_PLACEBOS, RESULTS, SEED, STATS,
                        WINDOW_S, group_cols)

jobs = pd.read_parquet(DATA / "jobs.parquet")
gpu = jobs[jobs.gpu].reset_index(drop=True)
flags = pd.read_parquet(DATA / "completeness.parquet", columns=["job_id", "complete", "complete_strict", "has_ganglia"])
inv = pd.read_csv(RESULTS / "channel_inventory.csv", index_col="channel")
used = inv.index[~inv.constant].tolist()
t0 = gpu.set_index("job_id")["t0_ms"]

## 1. Window series

For a channel with cadence *c*, interval *k* of a job holds the readings with
`t0 + k·c ≤ ts < t0 + (k+1)·c`. Inside an interval we first average each node, then average the
nodes. The time kept for the interval is that of its latest reading, so V1 checks real times.

In [2]:
def window_series(readings, cadence_s):
    r = readings[readings.job_id.isin(t0.index)].copy()
    r["bin"] = ((r.ts.values - t0.reindex(r.job_id).values) // (cadence_s * 1000)).astype(np.int16)
    per_node = r.groupby(["job_id", "node", "bin"], sort=False).agg(v=("value", "mean"), ts=("ts", "max"))
    s = per_node.groupby(level=["job_id", "bin"]).agg(value=("v", "mean"), ts=("ts", "max"),
                                                       n_nodes=("v", "size")).reset_index()
    s["value"] = s["value"].astype(np.float64)       # IPMI is stored as float32: the statistics are computed in float64
    return s


series = {}
for c in used:
    series[c] = window_series(pd.read_parquet(DATA / "raw_windows" / f"{c}.parquet"), int(inv.loc[c, "cadence_s"]))
    print(f"{c:15s} {len(series[c]):>11,} (job, interval) rows, {series[c].job_id.nunique():,} jobs")

mem_total           478,310 (job, interval) rows, 24,579 jobs


mem_free            630,584 (job, interval) rows, 24,420 jobs


mem_cached          629,485 (job, interval) rows, 24,375 jobs


mem_buffers         629,439 (job, interval) rows, 24,371 jobs


swap_free           629,504 (job, interval) rows, 24,374 jobs


cpu_wio             322,428 (job, interval) rows, 24,439 jobs


... (7 outputs left out)

gpu4_mem_temp     3,745,760 (job, interval) rows, 65,429 jobs


fan0_0            3,745,528 (job, interval) rows, 65,420 jobs


total_power       3,745,759 (job, interval) rows, 65,429 jobs


ambient           3,737,302 (job, interval) rows, 65,277 jobs


## 2. V1: no reading after `t0 + 20 min`

In [3]:
all_ts = pd.concat([s[["job_id", "ts"]] for s in series.values()])
rel = all_ts.ts.values - t0.reindex(all_ts.job_id).values
print(f"earliest reading: {rel.min() / 1000:.0f} s after t0;  latest: {rel.max() / 1000:.0f} s after t0")
assert rel.min() >= 0 and rel.max() < WINDOW_S * 1000, "V1 failed: a reading outside the window"
del all_ts, rel

earliest reading: 0 s after t0;  latest: 1199 s after t0


## 3. The ten groups, four statistics each

* `mem_active` = mean of `mem_total` in the window − `mem_free` − `mem_cached` − `mem_buffers`,
  per interval. The last three are sent together by ganglia (same timestamps); `mem_total` has its
  own cadence and is practically constant, so it enters through its window mean.
* `gpu_core_temp` and `gpu_mem_temp` average the GPU sensors in each interval.
* The other groups have one channel each.

The standard deviation is the population one (ddof = 0).

In [4]:
def group_series(g):
    if g == "mem_active":
        total = series["mem_total"].groupby("job_id").value.mean().rename("total")
        m = (series["mem_free"][["job_id", "bin", "value"]].rename(columns={"value": "free"})
             .merge(series["mem_cached"][["job_id", "bin", "value"]].rename(columns={"value": "cached"}), on=["job_id", "bin"])
             .merge(series["mem_buffers"][["job_id", "bin", "value"]].rename(columns={"value": "buffers"}), on=["job_id", "bin"])
             .merge(total, left_on="job_id", right_index=True))
        m["value"] = m.total - m.free - m.cached - m.buffers
        return m[["job_id", "bin", "value"]]
    parts = [series[c][["job_id", "bin", "value"]] for c in GROUPS[g]]
    return pd.concat(parts).groupby(["job_id", "bin"], as_index=False).value.mean()


telemetry = []
for g in GROUPS:
    v = group_series(g).groupby("job_id")["value"]
    telemetry.append(pd.DataFrame({f"{g}__min": v.min(), f"{g}__max": v.max(),
                                   f"{g}__mean": v.mean(), f"{g}__std": v.std(ddof=0)}))
telemetry = pd.concat(telemetry, axis=1)
print(f"{telemetry.shape[1]} telemetry columns (10 groups x 4 statistics)")

40 telemetry columns (10 groups x 4 statistics)


## 4. B0: the job request

B0 holds what the user requests at submission (P4): queue (`qos`) and `partition`, one-hot
encoded, plus `time_limit`, requested memory, nodes, cores, GPUs, tasks and cores per task. No
telemetry, no realized duration, no node identity, no collector-presence flag. The secondary
baseline of notebook 07 adds `has_ganglia`.

In [5]:
base = gpu[["job_id", "user_id", "end_time", "job_state", "target_a", "target_b", "qos", "partition",
            "num_nodes_alloc"] + B0_NUMERIC].merge(flags, on="job_id")
qos_values = sorted(base.qos.unique(), key=int)
partition_values = sorted(base.partition.astype(str).unique(), key=int)
for q in qos_values:
    base[f"b0_qos_{q}"] = (base.qos == q).astype(int)
for p in partition_values:
    base[f"b0_partition_{p}"] = (base.partition.astype(str) == p).astype(int)
for c in B0_NUMERIC:
    base[f"b0_{c}"] = base[c].astype(float)
B0 = [f"b0_qos_{q}" for q in qos_values] + [f"b0_partition_{p}" for p in partition_values] + [f"b0_{c}" for c in B0_NUMERIC]
print(f"B0 = {len(B0)} columns:", B0)

forbidden = ["runtime", "run_time", "end_time", "start_time", "has_ganglia", "user_id", "priority", "nodes_alloc"]
assert not any(f in c for c in B0 for f in forbidden), "B0 holds a forbidden column"

features = base.merge(telemetry, left_on="job_id", right_index=True, how="left")

B0 = 18 columns: ['b0_qos_1', 'b0_qos_2', 'b0_qos_3', 'b0_qos_4', 'b0_qos_7', 'b0_qos_8', 'b0_qos_9', 'b0_qos_11', 'b0_partition_0', 'b0_partition_1', 'b0_partition_2', 'b0_time_limit', 'b0_mem_req', 'b0_num_nodes_req', 'b0_num_cores_req', 'b0_num_gpus_req', 'b0_num_tasks', 'b0_cores_per_task']


### No missing values in the complete subset, nothing imputed

In [6]:
TEL = [c for g in GROUPS for c in group_cols(g)]
complete = features.complete.values == 1
nan_complete = features.loc[complete, TEL].isna().any(axis=1).mean()
print(f"complete subset: {nan_complete:.0%} of jobs have a missing telemetry feature")

# One job-request column has gaps: num_tasks is empty in the PM100 job table for some jobs. It enters
# the models as missing (all three models handle missing values natively); nothing is imputed.
print(features.loc[complete, B0].isna().mean()[lambda s: s > 0].map("{:.1%}".format).to_string())

# In the whole GPU cohort the ganglia groups are missing where the collector was off:
nan_cohort = {g: features[group_cols(g)].isna().any(axis=1).mean() for g in GROUPS}
print((100 * pd.Series(nan_cohort, name="% of GPU jobs with the group missing")).round(1).to_string())

complete subset: 0% of jobs have a missing telemetry feature
b0_num_tasks    8.7%
mem_active       63.2
swap_free        63.1
cpu_wio          63.0
proc_run         63.0
load_one         63.0
gpu_core_temp     0.9
gpu_mem_temp      0.9
fan               0.9
node_power        0.9
inlet_temp        1.1


## 5. Placebos

Placebo *k* (k = 0 … 19) takes the four columns of a real group and permutes the rows within the
population (P6). The same permutation is applied to the four columns, so the marginals and the
relation between the four statistics stay, and any link to the label is destroyed. The group
rotates: placebo *k* copies group number *k* mod 10 (or *k* mod 5 among the IPMI groups).

There is one family of 20 placebos per population, each permuted only within its population:

| population | rotates over | seed of placebo *k* | columns |
|---|---|---|---|
| complete (primary) | the 10 groups | 42 + *k* | `placebo_complete{k}__*` |
| GPU cohort (secondary) | the 5 IPMI groups | 1042 + *k* | `placebo_cohort{k}__*` |
| strict (sensitivity) | the 10 groups | 2042 + *k* | `placebo_strict{k}__*` |

In [7]:
def placebo(df, mask, k, groups, seed_base):
    src = df[group_cols(groups[k % len(groups)])].to_numpy(dtype=float)
    out = np.full_like(src, np.nan)
    rows = np.where(mask)[0]
    out[rows] = src[np.random.default_rng(seed_base + k).permutation(rows)]
    return out


families = {"complete": (features.complete.values == 1, list(GROUPS), SEED),
            "cohort": (np.ones(len(features), bool), IPMI_GROUPS, SEED + 1000),
            "strict": (features.complete_strict.values == 1, list(GROUPS), SEED + 2000)}
placebo_cols = {}
for pop, (mask, groups, seed) in families.items():
    for k in range(N_PLACEBOS):
        name = f"placebo_{pop}{k:02d}"
        values = placebo(features, mask, k, groups, seed)
        for j, stat in enumerate(STATS):
            placebo_cols[f"{name}__{stat}"] = values[:, j]
features = pd.concat([features, pd.DataFrame(placebo_cols, index=features.index)], axis=1)

# a placebo keeps the marginal distribution of its source group, in its population
m = features.complete == 1
assert np.allclose(np.sort(features.loc[m, "placebo_complete08__mean"]), np.sort(features.loc[m, "node_power__mean"]))

## Multi-node jobs

Telemetry of a multi-node job is averaged over its nodes. How much of the complete subset that
concerns:

In [8]:
comp = features[complete]
print(f"multi-node: {(comp.num_nodes_alloc > 1).mean():.1%} of complete jobs, "
      f"{(comp[comp.target_a == 1].num_nodes_alloc > 1).mean():.1%} of their target-(a) failures")

multi-node: 35.1% of complete jobs, 29.9% of their target-(a) failures


## Save

In [9]:
features.to_parquet(DATA / "features.parquet", index=False)
pd.Series(B0).to_csv(DATA / "b0_columns.csv", index=False, header=["column"])
print(f"data/features.parquet: {features.shape[0]:,} jobs x {features.shape[1]} columns")

data/features.parquet: 66,017 jobs x 317 columns
